# RAW to RGB — Part 2: Debayering

The sensor records only one colour per pixel arranged in a **Bayer pattern**:

```
R G R G ...
G B G B ...
R G R G ...
...........
```

To obtain a full-colour image we need to **interpolate** the two missing colour values at each pixel.
This process is called **debayering** or **demosaicing**.

In this notebook we implement **bilinear debayering** step by step.

In [ ]:
import numpy as np              # needed for image arrays
import HdM as HdM               # a useful function I prepared for you to display images pixel per pixel on your monitor
import imageio.v3 as iio        # Reading images with OpenImageIO
from scipy.ndimage import zoom  # Useful function to scale images

# The sRGB function introduced in the "Display Nonlinearity" script. We need this for correct display of linear data on your monitor
linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)

data = np.load('20_RawBayerImage_BlackSubtracted.npz')
raw_img_black_subtracted = data['raw_img_black_subtracted']
black_noise_std      = data['black_noise_std']

print(f'Loaded image shape (Dimensions in Pixels): {raw_img_black_subtracted.shape}')

## Look closely at the Bayer pattern

Zoom into a small crop — you should clearly see the RGGB mosaic.

In [ ]:
stops_above_white = 2
hOffset = 1000
vOffset = 1000
crop = raw_img_black_subtracted[vOffset:(vOffset+64), hOffset:(hOffset+64)]

HdM.show( linear2sRGB( np.clip( np.repeat( np.repeat(crop, 8, axis=0), 8, axis=1) * 2**stops_above_white, 0, 1)))
print('64×64 crop — individual Bayer pixels visible')


## Split the Bayer plane into 4 sub-images

The RGGB Bayer pattern maps to four sub-sampled images:

| c1 | c3 | c1 | c3 | ... |
|---|---|---|---|---|
| c2 | c4 | c2 | c4 | ... |
| c1 | c3 | c1 | c3 | ... |
| c2 | c4 | c2 | c4 | ... |
| ... | ... | ... | ... | ... |


In Python array indexing rows and columns start at 0, so `0::2` selects every second element starting at 0 (even), `1::2` selects odd.

In [ ]:
c1 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the top-left sensel of each 2x2 block.
c2 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the bottom-left sensel of each 2x2 block.
c3 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the top-right sensel of each 2x2 block.
c4 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the bottom-right sensel of each 2x2 block.

# Stick together all four images
composite_img = np.vstack([np.hstack([c1,c2]),np.hstack([c3,c4])])

print('Four Bayer sub-images — which two are green? Which one represents red, which one blue?')
HdM.show(linear2sRGB(np.clip(zoom(composite_img, 0.2) * 2**stops_above_white, 0, 1)))
print('Can you guess the green channels?')

## Identify the colour channels

The two green channels should look the most similar to each other.
We compute the **sum of absolute differences** between all channel pairs to find them programmatically.

In [ ]:
def sad(a, b):
    """Sum of absolute differences."""
        return  # Hinweis: Calculate the sum of absolute differences between a and b

# Print the sum of absolute differences for each pair:
for pair in [('c1','c2'), ('c1','c3'), ('c1','c4'), ('c2','c3'), ('c2','c4'), ('c3','c4')]:
    channels = {'c1': c1, 'c2': c2, 'c3': c3, 'c4': c4}
    print(f'SAD({pair[0]}, {pair[1]}) = {sad(channels[pair[0]], channels[pair[1]]):.0f}')

## Simple 1:4 combine (Canon C300 MkI-style)

The quickest approach: average the two green channels, keep R and B as-is.
This gives a half-resolution colour image.

In [ ]:
r =   # Hinweis: Assign the red channel to variable "r"
g =   # Hinweis: Calculate "g" as the mean of both green channels
b =   # Hinweis: Assign the blue channel to variable "b"

rgb_small = np.stack([r, g, b], axis=2)

print('Half-resolution colour image (direct channel combine)')
HdM.show(linear2sRGB(np.clip( zoom( rgb_small, [0.2,0.2,1] ) * 2**stops_above_white, 0, 1)))
print('Does the color checker and the skin tones look about right? But why is the image tinted green?')

## Full-resolution bilinear debayering

To keep full resolution we **interpolate** each missing colour value from its neighbours.

But before we start let's bring all images to the same color order by cropping:


In [ ]:
# As a next step you will implement the most basic 1:1 debayering algorithm: Bilinear Interpolation
# But before we start lets's bring all raw images to the same RGB pattern. Lets make the first pixel a blue sensitive sensel:

# If your blue channel is c1 uncomment and run the following command:
# raw = raw_img_black_subtracted[0:, 0:] # This actually does nothing

# If your blue channel is c2 uncomment and run the following command:
# raw = raw_img_black_subtracted[1:-1, 0:] # This crops the first and last row

# If your blue channel is c3 uncomment and run the following command:
raw = raw_img_black_subtracted[0:, 1:-1] # This crops the first and last column

# If your blue channel is c4 uncomment and run the following command:
# raw = raw_img_black_subtracted[1:-1, 1:-1] # This crops the first and last row and column

# Initalize four conveniently named image planes.
r  = raw[1::2,1::2]
g0 = raw[0::2,1::2]
g1 = raw[1::2,0::2]
b  = raw[0::2,0::2]

# Calculate the half size bayer image again to check if everything worked correctly
rgb_small = np.stack([r, (g0+g1)/2, b], axis=2)

print('Half-resolution colour image (direct channel combine). Should look exactly the same as your c1, c2, c3, c4 to r, g, b combination above.')
HdM.show(linear2sRGB(np.clip( zoom( rgb_small, [0.2,0.2,1] ) * 2**stops_above_white, 0, 1)))

print("The sizes of all vour sensel planes should be the same for the next step:\n", np.flip([r.shape, g0.shape, g1.shape, b.shape]))


In [ ]:
# For debugging start with a very small test sample: Comment out these lines if everything runs smoothly
raw = np.array([[1, 1, 3, 40],[5, 0, 2, 4],[7, 4, 1, 6],[20, 2, 8, 6]]) # Comment me out after debugging
r  = raw[1::2,1::2] # Comment me out after debugging
g0 = raw[0::2,1::2] # Comment me out after debugging
g1 = raw[1::2,0::2] # Comment me out after debugging
b  = raw[0::2,0::2] # Comment me out after debugging

# Lets first initialize the red image plane full of zeros:
r_lin_int = np.zeros((raw.shape[0]-2, raw.shape[1]-2), dtype=np.float32)

# The resulting linearly interpolated image plane 'r_lin_int' will be one sensel smaller,
# compared to 'raw' at each border, left, right, top bottom. The lost one sensel border
# is needed for interpolation. As a result the new red channel 'r_lin_int' starts with a
# red sensitive pixel. Hence we can directly write the red sensel values into the red image plane:
r_lin_int[0::2,0::2] =  r[0:-1, 0:-1]
r_lin_int[1::2,0::2] =   # Hinweis: Calculate by interpolation from 'r'
r_lin_int[0::2,1::2] =   # Hinweis: Calculate by interpolation from 'r'
r_lin_int[1::2,1::2] =   # Hinweis: Calculate by interpolation from 'r'

print("This is your full resolution reconstruction of the red color channel")
HdM.show(linear2sRGB(r_lin_int * 2**stops_above_white))

print(r_lin_int) # Comment me out after debugging



In [ ]:

# Initialize the green image plane full of zeros:
g_lin_int = np.zeros((raw.shape[0]-2, raw.shape[1]-2), dtype=np.float32)

# Next reconstruct the green channel by bilinear interploation:
g_lin_int[0::2,0::2] =   # Hinweis: Calculate by interpolation from 'g0 and g1'
g_lin_int[1::2,0::2] =   # Hinweis: Calculate by indexing from 'g0'
g_lin_int[0::2,1::2] =   # Hinweis: Calculate by indexing from 'g0'
g_lin_int[1::2,1::2] =   # Hinweis: Calculate by interpolation from 'g0 and g1'

print("This is your full resolution reconstruction of the green color channel")
HdM.show(linear2sRGB(np.clip(g_lin_int * 2**stops_above_white, 0, 1)))

print(g_lin_int) # Comment me out after debugging


In [ ]:
# Lets initialize the blue image plane full of zeros:
b_lin_int = np.zeros((raw.shape[0]-2, raw.shape[1]-2), dtype=np.float32)

# Next reconstruct the blue channel by bilinear interploation:
b_lin_int[0::2,0::2] =   # Hinweis: Calculate by interpolation from 'b'
b_lin_int[0::2,0::2] =   # Hinweis: Calculate by interpolation from 'b'
b_lin_int[0::2,0::2] =   # Hinweis: Calculate by interpolation from 'b'
b_lin_int[0::2,0::2] =   # Hinweis: Calculate by indexing from 'b'

print("This is your full resolution reconstruction of the blue color channel")
HdM.show(linear2sRGB(np.clip(b_lin_int * 2**stops_above_white, 0, 1)))

print(b_lin_int) # Comment me out after debugging

In [ ]:
# Finally, we can combine r,g and b to rgb and display it:

rgb_lin_int = np.stack([r_lin_int, g_lin_int, b_lin_int], axis=2)

HdM.show(linear2sRGB(np.clip(rgb_lin_int * 2**stops_above_white, 0, 1)))

# You made it. Congratulations!


## Compare: direct combine vs. bilinear interpolation

We crop a small region to compare resolution and edge sharpness.

In [ ]:
vo = 2260 // 2   # vertical offset (half-res coordinates for rgb_small)
ho = 2780 // 2   # horizontal offset (half-res coordinates for rgb_small)
crop_size = 100
stops = 2

# Crop from 4:1 half size Debayer
crop_small = rgb_small[vo:vo+crop_size, ho:ho+crop_size, :]

# Upsample for display by nearest-neighbour (zoom=2)
crop_small_up = np.repeat(np.repeat(crop_small, 2, axis=0), 2, axis=1)

# Crop from debayered image by bilinear interpolation
crop_full = rgb_lin_int[vo*2:(vo+crop_size)*2, ho*2:(ho+crop_size)*2, :]

print('4:1 combine (half-res) + upscale on the left compared to bilinear interpolation (full-res) on the right')
HdM.show( linear2sRGB( np.clip( np.hstack([crop_small_up,crop_full]) * 2**stops_above_white, 0, 1)))
print('Edges on the right side show zipper artifacts — fixed in next notebook with edge-directed debayering')


## Save result

In [ ]:
np.savez('21_RGB_BilinearDebayer.npz',
         rgb_lin_int=rgb_lin_int,
         raw=raw,
         black_noise_std=black_noise_std)

print('Saved: 21_RGB_BilinearDebayer.npz')